## 1. Setup

In [1]:
import os
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "font.family":       "sans-serif",
    "font.size":         11,
})
print("Libraries loaded.")

Libraries loaded.


## 2. Google Drive Setup

Dataset is cached to Google Drive after the first pull so the Open-Meteo API
is never called more than once per dataset version.


In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT  = "/content/drive/MyDrive/AgroGuard"
    USING_COLAB = True
    print("Google Drive mounted.")
except ModuleNotFoundError:
    DRIVE_ROOT  = "."
    USING_COLAB = False
    print("Not in Colab. Saving locally.")

os.makedirs(DRIVE_ROOT, exist_ok=True)
FEATURES_PATH = os.path.join(DRIVE_ROOT, "sprint1_features.csv")
print(f"Data root     : {DRIVE_ROOT}")
print(f"Features path : {FEATURES_PATH}")


Not in Colab. Saving locally.
Data root     : .
Features path : ./sprint1_features.csv


## 3. District Configuration

20 agricultural districts across the five countries in scope.
4 districts per country, selected for smallholder staple crop relevance.


In [3]:
DISTRICTS = [
    # Rwanda
    {"name": "Huye",      "country": "Rwanda",   "lat": -2.5967, "lon": 29.7397},
    {"name": "Musanze",   "country": "Rwanda",   "lat": -1.4985, "lon": 29.6328},
    {"name": "Nyamagabe", "country": "Rwanda",   "lat": -2.4733, "lon": 29.4978},
    {"name": "Rwamagana", "country": "Rwanda",   "lat": -1.9487, "lon": 30.4347},
    # Kenya
    {"name": "Nakuru",    "country": "Kenya",    "lat": -0.3031, "lon": 36.0800},
    {"name": "Kisumu",    "country": "Kenya",    "lat": -0.1022, "lon": 34.7617},
    {"name": "Eldoret",   "country": "Kenya",    "lat":  0.5143, "lon": 35.2698},
    {"name": "Kitale",    "country": "Kenya",    "lat":  1.0154, "lon": 35.0062},
    # Uganda
    {"name": "Mbarara",   "country": "Uganda",   "lat": -0.6072, "lon": 30.6545},
    {"name": "Gulu",      "country": "Uganda",   "lat":  2.7749, "lon": 32.2990},
    {"name": "Jinja",     "country": "Uganda",   "lat":  0.4244, "lon": 33.2042},
    {"name": "Mbale",     "country": "Uganda",   "lat":  1.0796, "lon": 34.1753},
    # Tanzania
    {"name": "Arusha",    "country": "Tanzania", "lat": -3.3869, "lon": 36.6830},
    {"name": "Mbeya",     "country": "Tanzania", "lat": -8.9094, "lon": 33.4607},
    {"name": "Dodoma",    "country": "Tanzania", "lat": -6.1730, "lon": 35.7395},
    {"name": "Morogoro",  "country": "Tanzania", "lat": -6.8242, "lon": 37.6556},
    # Ethiopia
    {"name": "Hawassa",   "country": "Ethiopia", "lat":  7.0621, "lon": 38.4767},
    {"name": "Jimma",     "country": "Ethiopia", "lat":  7.6790, "lon": 36.8344},
    {"name": "Bahir Dar", "country": "Ethiopia", "lat": 11.5742, "lon": 37.3614},
    {"name": "Dire Dawa", "country": "Ethiopia", "lat":  9.5931, "lon": 41.8661},
]

START_DATE = "2021-01-01"
END_DATE   = "2023-12-31"

by_country = pd.Series([d["country"] for d in DISTRICTS]).value_counts()
print(f"Total districts : {len(DISTRICTS)}")
print(f"Countries       : {dict(by_country)}")
print(f"Date range      : {START_DATE} to {END_DATE}")
print(f"Expected rows   : ~{len(DISTRICTS) * 1095:,}")

Total districts : 20
Countries       : {'Rwanda': 4, 'Kenya': 4, 'Uganda': 4, 'Tanzania': 4, 'Ethiopia': 4}
Date range      : 2021-01-01 to 2023-12-31
Expected rows   : ~21,900


## 4. Data Collection

Weather comes from the Open-Meteo ERA5 archive (no API key required). Set
`USE_SYNTHETIC = False` in the next cell to pull real historical data; leave it
`True` for an offline demo run.

Caches are namespaced by source (`weather_cache_synthetic.csv` versus
`weather_cache_era5.csv`), so switching modes never mixes the two. The real pull
is resumable: each district is checkpointed to its own file under `raw_era5/` as
it downloads, so a dropped connection resumes where it left off instead of
restarting from scratch. Requests are rate-limited and retried with exponential
backoff.


In [4]:
# Switch to False to pull real historical weather from the Open-Meteo ERA5 archive.
USE_SYNTHETIC = False

# Cache and raw files are namespaced by source so switching modes never mixes
# synthetic and real data.
DATA_SOURCE   = "synthetic" if USE_SYNTHETIC else "era5"
WEATHER_CACHE = os.path.join(DRIVE_ROOT, f"weather_cache_{DATA_SOURCE}.csv")
RAW_DIR       = os.path.join(DRIVE_ROOT, f"raw_{DATA_SOURCE}")


def fetch_weather_api(district: dict, start: str, end: str,
                      retries: int = 3, backoff: float = 2.0) -> pd.DataFrame:
    """
    Pull daily weather from the Open-Meteo ERA5 archive, plus hourly relative
    humidity aggregated to a daily mean. Rate-limited to one request per 0.5s and
    retried with exponential backoff on transient network or rate-limit errors.
    """
    url = "https://archive-api.open-meteo.com/v1/era5"
    params = {
        "latitude":   district["lat"],
        "longitude":  district["lon"],
        "start_date": start,
        "end_date":   end,
        "daily":  ["temperature_2m_max", "temperature_2m_min",
                   "precipitation_sum", "wind_speed_10m_max"],
        "hourly": ["relative_humidity_2m"],
        "timezone": "Africa/Nairobi",
    }

    payload = None
    for attempt in range(1, retries + 1):
        try:
            time.sleep(0.5)
            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()
            payload = response.json()
            break
        except (requests.RequestException, ValueError) as error:
            if attempt == retries:
                raise RuntimeError(
                    f"Open-Meteo fetch failed for {district['name']} after "
                    f"{retries} attempts: {error}"
                )
            time.sleep(backoff * attempt)

    df = pd.DataFrame(payload["daily"])
    df["date"] = pd.to_datetime(df["time"])

    # Aggregate hourly relative humidity to a daily mean and join on date.
    hourly = pd.DataFrame(payload["hourly"])
    hourly["date"] = pd.to_datetime(hourly["time"]).dt.normalize()
    daily_humidity = hourly.groupby("date")["relative_humidity_2m"].mean().round(1)
    df["humidity"] = df["date"].map(daily_humidity)

    df["district"] = district["name"]
    df["country"]  = district["country"]
    df["lat"]      = district["lat"]
    df["lon"]      = district["lon"]
    return df.drop(columns=["time"]).rename(columns={
        "temperature_2m_max": "temp_max",
        "temperature_2m_min": "temp_min",
        "precipitation_sum":  "rainfall",
        "wind_speed_10m_max": "wind_max",
    })


def generate_synthetic_weather(district: dict, start: str, end: str) -> pd.DataFrame:
    """
    Realistic synthetic weather matching the Open-Meteo schema, including a
    relative humidity column. Bimodal rainy seasons (Mar-May, Oct-Dec) typical
    of East Africa. Used for local demos. Swap to fetch_weather_api() for production.
    """
    dates = pd.date_range(start=start, end=end, freq="D")
    n     = len(dates)
    doy   = np.array([d.timetuple().tm_yday for d in dates]) / 365.0

    base = {"Rwanda": 22.0, "Kenya": 19.5, "Uganda": 21.0,
            "Tanzania": 20.0, "Ethiopia": 18.5}.get(district["country"], 21.0)

    temp_max = base + 4 * np.sin(2 * np.pi * doy) + np.random.normal(0, 1.2, n)
    temp_min = temp_max - 8 + np.random.normal(0, 0.8, n)

    s1 = 12 * np.exp(-0.5 * ((doy * 365 - 120) / 30) ** 2)
    s2 = 10 * np.exp(-0.5 * ((doy * 365 - 300) / 35) ** 2)
    rainfall = np.maximum(0, np.random.exponential(s1 + s2 + 0.3, n))
    rainfall[np.random.random(n) < 0.45] = 0

    # Relative humidity rises with rainfall and the rainy seasons, plus noise.
    humidity = 60 + 0.8 * np.minimum(rainfall, 25) + 0.6 * (s1 + s2) + np.random.normal(0, 4, n)
    humidity = np.clip(humidity, 30, 100)

    return pd.DataFrame({
        "date":     dates,
        "district": district["name"],
        "country":  district["country"],
        "lat":      district["lat"],
        "lon":      district["lon"],
        "temp_max": np.round(temp_max, 1),
        "temp_min": np.round(temp_min, 1),
        "rainfall": np.round(rainfall, 1),
        "wind_max": np.round(np.abs(np.random.normal(18, 5, n)), 1),
        "humidity": np.round(humidity, 1),
    })


print(f"Mode       : {'Synthetic (demo)' if USE_SYNTHETIC else 'Open-Meteo ERA5 (real)'}")
print(f"Cache file : {WEATHER_CACHE}")


Mode       : Open-Meteo ERA5 (real)
Cache file : ./weather_cache_era5.csv


In [5]:
if os.path.exists(WEATHER_CACHE):
    print(f"Combined cache found: {WEATHER_CACHE}")
    weather_raw = pd.read_csv(WEATHER_CACHE, parse_dates=["date"])
    print(f"Loaded {len(weather_raw):,} rows.")
else:
    print(f"No combined cache. Building the {DATA_SOURCE} dataset...")
    load_fn = generate_synthetic_weather if USE_SYNTHETIC else fetch_weather_api
    os.makedirs(RAW_DIR, exist_ok=True)

    frames = []
    for i, district in enumerate(DISTRICTS, 1):
        part_path = os.path.join(RAW_DIR, f"{district['name']}.csv")
        tag = f"[{i:02d}/{len(DISTRICTS)}] {district['name']:<10} {district['country']}"
        if os.path.exists(part_path):
            df = pd.read_csv(part_path, parse_dates=["date"])
            print(f"{tag}: cached  ({len(df):,} rows).")
        else:
            df = load_fn(district, START_DATE, END_DATE)
            df.to_csv(part_path, index=False)   # checkpoint so the pull is resumable
            print(f"{tag}: fetched ({len(df):,} rows).")
        frames.append(df)

    weather_raw = pd.concat(frames, ignore_index=True)
    weather_raw.to_csv(WEATHER_CACHE, index=False)
    print(f"\nSaved combined cache: {WEATHER_CACHE}")

quality_cols = [c for c in ["temp_max", "temp_min", "rainfall", "humidity"]
                if c in weather_raw.columns]
print(f"\nTotal rows    : {len(weather_raw):,}")
print(f"Districts     : {weather_raw['district'].nunique()} (expected {len(DISTRICTS)})")
print(f"Countries     : {weather_raw['country'].nunique()}")
print(f"Date range    : {weather_raw['date'].min().date()} to {weather_raw['date'].max().date()}")
print(f"Missing values: {int(weather_raw[quality_cols].isna().sum().sum())}")
weather_raw.head(8)


No combined cache. Building the era5 dataset...


[01/20] Huye       Rwanda: fetched (1,095 rows).


[02/20] Musanze    Rwanda: fetched (1,095 rows).


[03/20] Nyamagabe  Rwanda: fetched (1,095 rows).


[04/20] Rwamagana  Rwanda: fetched (1,095 rows).


...

[17/20] Hawassa    Ethiopia: fetched (1,095 rows).


[18/20] Jimma      Ethiopia: fetched (1,095 rows).


[19/20] Bahir Dar  Ethiopia: fetched (1,095 rows).


[20/20] Dire Dawa  Ethiopia: fetched (1,095 rows).

Saved combined cache: ./weather_cache_era5.csv

Total rows    : 21,900
Districts     : 20 (expected 20)
Countries     : 5
Date range    : 2021-01-01 to 2023-12-31
Missing values: 0


,temp_max,temp_min,rainfall,wind_max,date,humidity,district,country,lat,lon
0,20.7,15.3,6.7,8.9,2021-01-01,87.4,Huye,Rwanda,-2.5967,29.7397
1,22.9,13.6,12.7,11.5,2021-01-02,80.7,Huye,Rwanda,-2.5967,29.7397
2,23.4,15.5,1.4,8.0,2021-01-03,79.4,Huye,Rwanda,-2.5967,29.7397
3,23.5,14.3,3.5,12.4,2021-01-04,79.5,Huye,Rwanda,-2.5967,29.7397
4,23.2,13.9,1.1,11.2,2021-01-05,76.5,Huye,Rwanda,-2.5967,29.7397
5,24.1,15.9,0.6,12.9,2021-01-06,74.1,Huye,Rwanda,-2.5967,29.7397
6,24.6,12.5,0.1,12.2,2021-01-07,68.6,Huye,Rwanda,-2.5967,29.7397
7,24.7,14.5,11.1,9.5,2021-01-08,74.2,Huye,Rwanda,-2.5967,29.7397


## 5. Data Cleaning

In [6]:
def quality_report(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in ["temp_max", "temp_min", "rainfall", "wind_max", "humidity"]
            if c in df.columns]
    return pd.DataFrame([{
        "column":      c,
        "missing":     df[c].isna().sum(),
        "missing_pct": round(df[c].isna().mean() * 100, 2),
        "min":         round(df[c].min(), 2),
        "max":         round(df[c].max(), 2),
        "mean":        round(df[c].mean(), 2),
    } for c in cols])

quality_report(weather_raw)


,column,missing,missing_pct,min,max,mean
0,temp_max,0,0.0,15.6,37.0,25.96
1,temp_min,0,0.0,7.0,23.5,15.78
2,rainfall,0,0.0,0.0,99.9,2.84
3,wind_max,0,0.0,4.0,44.3,14.39
4,humidity,0,0.0,10.5,97.0,68.09


In [7]:
numeric_cols = [c for c in ["temp_max", "temp_min", "rainfall", "wind_max", "humidity"]
                if c in weather_raw.columns]
weather = weather_raw.sort_values(["district", "date"]).copy()

weather[numeric_cols] = (
    weather
    .groupby("district")[numeric_cols]
    .transform(lambda col: col.ffill().bfill())
)
weather["rainfall"] = weather["rainfall"].clip(lower=0)

print(f"Missing values after cleaning : {weather[numeric_cols].isna().sum().sum()}")
print(f"Rows retained                 : {len(weather):,}")


Missing values after cleaning : 0
Rows retained                 : 21,900


## 6. Feature Engineering

All four features are computed per district independently.

| Feature | Window | Rationale |
|---|---|---|
| `consecutive_wet_days` | Rolling | Sustained moisture drives spore germination |
| `temp_spread_7d` | 7 days | Diurnal stress weakens plant defences |
| `humidity_deviation` | 7-day mean | Overnight condensation activates pathogens |
| `rainfall_7d_total` | 7 days | Cumulative weekly rainfall sustains canopy wetness |


In [8]:
def consecutive_wet_days(rainfall: pd.Series, threshold: float = 1.0) -> pd.Series:
    """Count consecutive days with rainfall >= threshold mm ending on each date."""
    values, result, count = rainfall.values, np.zeros(len(rainfall), dtype=int), 0
    for i, rain in enumerate(values):
        count     = count + 1 if rain >= threshold else 0
        result[i] = count
    return pd.Series(result, index=rainfall.index)


def estimate_humidity(temp_max: pd.Series, temp_min: pd.Series) -> pd.Series:
    """
    Estimate relative humidity via the Magnus dew point formula. Used only as a
    fallback when measured relative humidity is not present in the data.
    """
    alpha     = (17.27 * temp_min) / (237.7 + temp_min) + np.log(0.75)
    dew_point = (237.7 * alpha) / (17.27 - alpha)
    rh = 100 * (np.exp((17.625 * dew_point) / (243.04 + dew_point)) /
                np.exp((17.625 * temp_max)   / (243.04 + temp_max)))
    return rh.clip(0, 100)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all four feature engineering steps. Input must be sorted by date."""
    df = df.copy()
    df["consecutive_wet_days"] = consecutive_wet_days(df["rainfall"])
    df["temp_spread_7d"]       = (df["temp_max"] - df["temp_min"]).rolling(7, min_periods=1).mean().round(2)

    # Prefer measured relative humidity; fall back to the dew point estimate.
    if "humidity" in df.columns and df["humidity"].notna().any():
        humidity = df["humidity"]
    else:
        humidity = estimate_humidity(df["temp_max"], df["temp_min"])
    df["humidity_deviation"] = (humidity - humidity.rolling(7, min_periods=1).mean()).round(2)

    df["rainfall_7d_total"] = df["rainfall"].rolling(7, min_periods=1).sum().round(2)
    return df


print("Feature functions defined.")


Feature functions defined.


In [9]:
frames = []
for district_name in weather["district"].unique():
    subset = weather[weather["district"] == district_name].copy()
    frames.append(engineer_features(subset))

weather_features = pd.concat(frames, ignore_index=True)

print(f"Features engineered. Total rows: {len(weather_features):,}")
weather_features[["date", "district", "country", "rainfall",
                   "consecutive_wet_days", "temp_spread_7d",
                   "humidity_deviation"]].head(10)

Features engineered. Total rows: 21,900


,date,district,country,rainfall,consecutive_wet_days,temp_spread_7d,humidity_deviation
0,2021-01-01,Arusha,Tanzania,23.2,1,4.20,0.00
1,2021-01-02,Arusha,Tanzania,5.5,2,6.05,-4.00
2,2021-01-03,Arusha,Tanzania,1.8,3,6.90,-0.87
3,2021-01-04,Arusha,Tanzania,0.8,0,7.93,-6.88
4,2021-01-05,Arusha,Tanzania,0.3,0,8.90,-12.22
5,2021-01-06,Arusha,Tanzania,0.7,0,9.53,-9.60
6,2021-01-07,Arusha,Tanzania,0.2,0,9.97,-13.03
7,2021-01-08,Arusha,Tanzania,0.2,0,11.09,-5.41
8,2021-01-09,Arusha,Tanzania,4.4,1,11.29,1.20
9,2021-01-10,Arusha,Tanzania,1.6,2,10.84,13.27


## 7. Disease Risk Labelling from Mendeley Data

The synthetic target used during early prototyping is replaced here with labels
grounded in real field evidence: the Mduma and Mayo (2024) maize imagery dataset
for Tanzania (Mendeley Data, fkw49mz3xs).

That dataset contains 9,356 labelled maize leaf images collected in Arusha across
the 2022 long rains (March to May) and short rains (October to December), split
into three classes: Healthy, Maize Lethal Necrosis (MLN), and Maize Streak Virus
(MSV). It carries no dates or coordinates, only the per-class image counts implied
by its folder structure.

### Labelling strategy

We cannot map individual images to individual days, so the dataset is used at the
level it actually supports, the class distribution, combined with the Tanzanian
weather records:

1. **Observed prevalence.** The diseased fraction (MLN + MSV) / total sets how
   often HIGH risk occurred during the documented disease seasons.
2. **Season window.** Labelling is restricted to records from the labelled
   districts inside the two 2022 rainy seasons the images were collected in
   (the districts are listed in the next cell).
3. **Weather conduciveness.** Within those windows each day is scored by a
   transparent index built from the three engineered features. Disease occurrence
   is then drawn as a Bernoulli outcome whose probability rises with that index
   and is calibrated so the season mean equals the observed prevalence.

This is weak supervision: the images tell us how much disease was present, the
weather tells us which days were most conducive. The Bernoulli draw keeps the
labels realistically noisy rather than a hard threshold, so the model has a real
signal to learn. The curated dataset over-represents diseased leaves relative to a
random field survey, so prevalence is treated as a seasonal signal, not an
absolute incidence rate.


In [10]:
# ---------------------------------------------------------------------------
# Disease class distribution from the Mduma and Mayo (2024) Mendeley dataset.
# Source: https://data.mendeley.com/datasets/fkw49mz3xs/1
#
# Counts are read directly from the dataset folder structure when the extracted
# archive is available on Drive. Otherwise the published per-class counts below
# are used so the notebook stays reproducible without the image files present.
# ---------------------------------------------------------------------------

# Published counts for version 1 of the dataset (one folder per class).
PUBLISHED_CLASS_COUNTS = {"Healthy": 3073, "MLN": 3231, "MSV": 3052}

MENDELEY_IMAGE_ROOT = os.path.join(DRIVE_ROOT, "mendeley_maize")
IMAGE_EXTENSIONS    = {".jpg", ".jpeg", ".png"}


def count_images_per_class(root: str) -> dict:
    """Count image files inside each class subfolder of the dataset archive."""
    counts = {}
    for entry in sorted(os.listdir(root)):
        class_dir = os.path.join(root, entry)
        if os.path.isdir(class_dir):
            counts[entry] = sum(
                1 for f in os.listdir(class_dir)
                if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS
            )
    return counts


if os.path.isdir(MENDELEY_IMAGE_ROOT):
    DISEASE_CLASS_COUNTS = count_images_per_class(MENDELEY_IMAGE_ROOT)
    print(f"Class counts read from dataset folders at {MENDELEY_IMAGE_ROOT}")
else:
    DISEASE_CLASS_COUNTS = dict(PUBLISHED_CLASS_COUNTS)
    print("Dataset folders not found. Using published class counts.")

DISEASED_CLASSES   = ["MLN", "MSV"]
total_images       = sum(DISEASE_CLASS_COUNTS.values())
diseased_images    = sum(DISEASE_CLASS_COUNTS[c] for c in DISEASED_CLASSES)
disease_prevalence = diseased_images / total_images

print(f"Class counts       : {DISEASE_CLASS_COUNTS}")
print(f"Total images       : {total_images:,}")
print(f"Diseased (MLN+MSV) : {diseased_images:,}")
print(f"Disease prevalence : {disease_prevalence:.1%}")


Dataset folders not found. Using published class counts.
Class counts       : {'Healthy': 3073, 'MLN': 3231, 'MSV': 3052}
Total images       : 9,356
Diseased (MLN+MSV) : 6,283
Disease prevalence : 67.2%


### Extending labels to Mbeya and Morogoro

The Mendeley images were collected in Arusha, but labelling Arusha alone leaves
only about 184 training rows (one district across two 92-day seasons), too few for
a stable model. To widen the labelled set without inventing data, the same seasonal
prevalence is extended to two further districts already in scope: Mbeya and Morogoro.

Both host Tanzania Agricultural Research Institute (TARI) stations, sit in the
same country as Arusha, and follow the same 2022 long and short rains calendar.
That shared agronomic and seasonal context makes it defensible to assume a
comparable seasonal disease prevalence (67.2%) in those districts during the same
windows. The calibrated Bernoulli labelling is applied to each district
independently, ranking its own days by weather conduciveness. This triples the
labelled training rows to roughly 550 (about 184 per district).

The conduciveness index used for labelling keeps the original three features, so
the labels stay directly comparable to the Arusha-only version. The new
`rainfall_7d_total` feature is used for modelling, not for label generation.


In [11]:
# Districts with labelled training data. Arusha is the source of the Mendeley
# images; Mbeya and Morogoro host TARI stations in the same country and the same
# 2022 seasons, so the observed seasonal prevalence is extended to them.
LABELLED_DISTRICTS = ["Arusha", "Mbeya", "Morogoro"]

# 2022 growing seasons the dataset images were collected in.
DISEASE_SEASONS = [
    ("2022-03-01", "2022-05-31"),  # long rains  (Masika)
    ("2022-10-01", "2022-12-31"),  # short rains (Vuli)
]

# Steepness of the conduciveness-to-probability curve. A value of 1.0 keeps the
# labels realistically noisy so the model learns a gradient rather than a hard
# threshold; larger values tie the label more tightly to the weather score.
DISEASE_LABEL_BETA = 1.0
LABEL_RANDOM_STATE = 42


def conduciveness_index(df: pd.DataFrame) -> pd.Series:
    """
    Standardised sum of the three core features. Higher means weather more
    favourable to disease: sustained moisture, wide diurnal swings, elevated
    overnight humidity. The new rainfall_7d_total feature is intentionally left
    out so labels match the original Arusha-only definition.
    """
    z = lambda s: (s - s.mean()) / (s.std() + 1e-9)
    return (z(df["consecutive_wet_days"])
            + z(df["temp_spread_7d"])
            + z(df["humidity_deviation"]))


def calibrated_intercept(logits: np.ndarray, target_rate: float) -> float:
    """Bisection search for offset c so that mean(sigmoid(logits + c)) == target_rate."""
    sigmoid = lambda v: 1.0 / (1.0 + np.exp(-v))
    low, high = -20.0, 20.0
    for _ in range(60):
        mid = (low + high) / 2.0
        if sigmoid(logits + mid).mean() < target_rate:
            low = mid
        else:
            high = mid
    return (low + high) / 2.0


def label_disease_risk(df: pd.DataFrame, prevalence: float, seed: int) -> pd.DataFrame:
    """
    Assign calibrated Bernoulli HIGH-risk labels to one district's season records.
    Probability rises with weather conduciveness and is calibrated so the mean
    matches the observed disease prevalence.
    """
    df = df.copy()
    score   = conduciveness_index(df)
    z_score = (score - score.mean()) / (score.std() + 1e-9)
    logits  = DISEASE_LABEL_BETA * z_score.values
    logits += calibrated_intercept(logits, prevalence)
    probabilities = 1.0 / (1.0 + np.exp(-logits))
    rng = np.random.RandomState(seed)
    df["disease_risk"] = (rng.random(len(df)) < probabilities).astype(int)
    return df


# Build the season mask once, then label each district independently so every
# district is calibrated to the same prevalence on its own weather record.
season_mask = np.zeros(len(weather_features), dtype=bool)
for start, end in DISEASE_SEASONS:
    season_mask |= weather_features["date"].between(start, end).values

labelled_frames = []
for offset, district in enumerate(LABELLED_DISTRICTS):
    subset = weather_features[(weather_features["district"] == district) & season_mask].copy()
    labelled_frames.append(
        label_disease_risk(subset, disease_prevalence, LABEL_RANDOM_STATE + offset)
    )

labelled = pd.concat(labelled_frames, ignore_index=True)

print(f"Labelled districts : {', '.join(LABELLED_DISTRICTS)}")
print(f"Labelled records   : {len(labelled):,}")
for district in LABELLED_DISTRICTS:
    d = labelled[labelled["district"] == district]
    print(f"  {district:10s}: {len(d):3d} rows, HIGH-risk {d['disease_risk'].mean():.1%}")
print(f"Target prevalence  : {disease_prevalence:.1%}")
print(f"Overall HIGH-risk  : {labelled['disease_risk'].mean():.1%}")


Labelled districts : Arusha, Mbeya, Morogoro
Labelled records   : 552
  Arusha    : 184 rows, HIGH-risk 67.4%
  Mbeya     : 184 rows, HIGH-risk 65.8%
  Morogoro  : 184 rows, HIGH-risk 67.9%
Target prevalence  : 67.2%
Overall HIGH-risk  : 67.0%


## 8. Model Training and Evaluation

Two models are trained on the labelled Tanzanian records (Arusha, Mbeya, and
Morogoro): a logistic regression baseline and an XGBoost classifier. All four
engineered features are used.

A missed outbreak warning costs far more than a false alarm, so **recall on the
HIGH-risk class is the priority metric**. Both models are weighted toward the
positive class (balanced weights for logistic regression, `scale_pos_weight` for
XGBoost).

With only a few hundred labelled rows a single train-test split gives a noisy
estimate, so evaluation uses **stratified 5-fold cross-validation**, reporting the
mean and standard deviation of HIGH-risk recall and precision across folds. The
final models used for the all-district prediction are then refit on all labelled
rows.


In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score
from xgboost import XGBClassifier

FEATURE_COLS = ["consecutive_wet_days", "temp_spread_7d",
                "humidity_deviation", "rainfall_7d_total"]
TARGET_COL   = "disease_risk"

X = labelled[FEATURE_COLS].values
y = labelled[TARGET_COL].values

N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)


def build_models(y_train: np.ndarray) -> dict:
    """Instantiate both models with recall-oriented class weighting."""
    neg, pos         = np.bincount(y_train)
    scale_pos_weight = neg / pos
    logistic = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        verbosity=0,
    )
    return {"Logistic Regression": logistic, "XGBoost": xgb}


print(f"Labelled records : {len(X):,}")
print(f"Features         : {', '.join(FEATURE_COLS)}")
print(f"HIGH-risk share  : {y.mean():.1%}")
print(f"Evaluation       : stratified {N_SPLITS}-fold cross-validation")


Labelled records : 552
Features         : consecutive_wet_days, temp_spread_7d, humidity_deviation, rainfall_7d_total
HIGH-risk share  : 67.0%
Evaluation       : stratified 5-fold cross-validation


In [13]:
MODEL_NAMES = ["Logistic Regression", "XGBoost"]
fold_scores = {name: {"recall": [], "precision": []} for name in MODEL_NAMES}

for train_idx, test_idx in cv.split(X, y):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    # Scaler is fit inside each fold to keep the held-out rows fully unseen.
    fold_scaler = StandardScaler().fit(X_tr)
    models      = build_models(y_tr)

    for name, model in models.items():
        if name == "Logistic Regression":
            model.fit(fold_scaler.transform(X_tr), y_tr)
            preds = model.predict(fold_scaler.transform(X_te))
        else:
            model.fit(X_tr, y_tr)
            preds = model.predict(X_te)
        fold_scores[name]["recall"].append(recall_score(y_te, preds, zero_division=0))
        fold_scores[name]["precision"].append(precision_score(y_te, preds, zero_division=0))

print(f"Stratified {N_SPLITS}-fold cross-validation (HIGH-risk class)")
print("=" * 55)
for name in MODEL_NAMES:
    recall = np.array(fold_scores[name]["recall"])
    prec   = np.array(fold_scores[name]["precision"])
    print(f"\n{name}")
    print(f"  Recall    : {recall.mean():.3f} +/- {recall.std():.3f}")
    print(f"  Precision : {prec.mean():.3f} +/- {prec.std():.3f}")


Stratified 5-fold cross-validation (HIGH-risk class)

Logistic Regression
  Recall    : 0.659 +/- 0.074
  Precision : 0.787 +/- 0.036

XGBoost
  Recall    : 0.727 +/- 0.056
  Precision : 0.760 +/- 0.026


In [14]:
# Refit both models on all labelled rows. The XGBoost model is the one applied to
# every district in the next section; the scaler is kept for the logistic baseline.
scaler       = StandardScaler().fit(X)
final_models = build_models(y)

lr_model = final_models["Logistic Regression"]
lr_model.fit(scaler.transform(X), y)

xgb_model = final_models["XGBoost"]
xgb_model.fit(X, y)

print("Final models refit on all labelled records.")
print(f"Training rows : {len(X):,}")
print(f"Features      : {', '.join(FEATURE_COLS)}")


Final models refit on all labelled records.
Training rows : 552
Features      : consecutive_wet_days, temp_spread_7d, humidity_deviation, rainfall_7d_total


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Cross-Validated Model Evaluation (HIGH-risk class)",
             fontsize=13, fontweight="bold")

recall_mean = [np.mean(fold_scores[m]["recall"])    for m in MODEL_NAMES]
recall_std  = [np.std(fold_scores[m]["recall"])     for m in MODEL_NAMES]
prec_mean   = [np.mean(fold_scores[m]["precision"]) for m in MODEL_NAMES]
prec_std    = [np.std(fold_scores[m]["precision"])  for m in MODEL_NAMES]

x     = np.arange(len(MODEL_NAMES))
width = 0.35
axes[0].bar(x - width/2, recall_mean, width, yerr=recall_std, capsize=4,
            label="Recall (HIGH)", color="#C0504D")
axes[0].bar(x + width/2, prec_mean, width, yerr=prec_std, capsize=4,
            label="Precision (HIGH)", color="#2E75B6")
axes[0].set_xticks(x)
axes[0].set_xticklabels(MODEL_NAMES)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Score")
axes[0].set_title(f"Mean and std over {N_SPLITS} folds")
axes[0].legend(frameon=False)

importances = xgb_model.feature_importances_
order       = np.argsort(importances)
axes[1].barh(np.array(FEATURE_COLS)[order], importances[order], color="#70AD47")
axes[1].set_xlabel("Importance Score")
axes[1].set_title("XGBoost Feature Importance")

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()


### Feature Diagnostics and Validation

Two checks probe how trustworthy these results are, given that the labels are
weakly derived rather than field-observed.

**Collinearity.** `rainfall_7d_total` and `consecutive_wet_days` both track wet
spells, so the correlation matrix and a permutation-importance pass (which is
robust to correlated inputs) confirm whether the new feature contributes signal
of its own rather than echoing an existing one.

**Robustness and transfer.** The Bernoulli labels depend on a random draw, so
recall is recomputed across several label seeds to confirm it is stable, not an
artifact of one draw. A leave-one-district-out test then trains on two districts
and measures recall on the third, probing whether the weather-to-risk
relationship transfers across districts.

These checks test internal stability and cross-district transfer, not absolute
field accuracy. No ground-truth outbreak records exist for these districts, so
true validation still requires field surveys. That remains the key open item for
Sprint 2 and is the honest ceiling on what these metrics can claim.


In [16]:
from sklearn.inspection import permutation_importance

# Correlation among the four model features on the labelled set.
feature_corr = labelled[FEATURE_COLS].corr().round(2)
print("Feature correlation matrix (labelled records):")
print(feature_corr)

# Permutation importance is robust to the collinearity that gain-based importance
# can distort. It measures the recall drop when each feature is shuffled, showing
# whether rainfall_7d_total carries signal beyond consecutive_wet_days.
perm = permutation_importance(xgb_model, X, y, scoring="recall",
                              n_repeats=20, random_state=42)
perm_df = (
    pd.DataFrame({"feature":    FEATURE_COLS,
                  "importance": perm.importances_mean.round(4),
                  "std":        perm.importances_std.round(4)})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("\nPermutation importance (drop in recall when shuffled):")
print(perm_df)


Feature correlation matrix (labelled records):
                      consecutive_wet_days  temp_spread_7d  \
consecutive_wet_days                  1.00           -0.52   
temp_spread_7d                       -0.52            1.00   
humidity_deviation                    0.16            0.06   
rainfall_7d_total                     0.74           -0.68   

                      humidity_deviation  rainfall_7d_total  
consecutive_wet_days                0.16               0.74  
temp_spread_7d                      0.06              -0.68  
humidity_deviation                  1.00               0.07  
rainfall_7d_total                   0.07               1.00  

Permutation importance (drop in recall when shuffled):
                feature  importance     std
0        temp_spread_7d      0.1607  0.0157
1    humidity_deviation      0.1341  0.0146
2     rainfall_7d_total      0.1035  0.0106
3  consecutive_wet_days      0.0451  0.0081


In [17]:
def label_all_districts(global_seed: int):
    """Label all three districts with a seed offset and return (X, y)."""
    frames = []
    for offset, district in enumerate(LABELLED_DISTRICTS):
        subset = weather_features[(weather_features["district"] == district) & season_mask].copy()
        frames.append(label_disease_risk(subset, disease_prevalence, global_seed + offset))
    data = pd.concat(frames, ignore_index=True)
    return data[FEATURE_COLS].values, data[TARGET_COL].values


def cv_xgb_recall(X_arr: np.ndarray, y_arr: np.ndarray) -> float:
    """Mean HIGH-risk recall for XGBoost under the shared CV splitter."""
    scores = []
    for train_idx, test_idx in cv.split(X_arr, y_arr):
        model = build_models(y_arr[train_idx])["XGBoost"]
        model.fit(X_arr[train_idx], y_arr[train_idx])
        scores.append(recall_score(y_arr[test_idx], model.predict(X_arr[test_idx]),
                                    zero_division=0))
    return float(np.mean(scores))


# Re-label and re-evaluate across ten independent label draws.
seed_recalls = np.array([cv_xgb_recall(*label_all_districts(s)) for s in range(42, 52)])

print("XGBoost HIGH-risk recall across 10 label seeds")
print("=" * 47)
print(f"  mean  : {seed_recalls.mean():.3f}")
print(f"  std   : {seed_recalls.std():.3f}")
print(f"  range : {seed_recalls.min():.3f} to {seed_recalls.max():.3f}")


XGBoost HIGH-risk recall across 10 label seeds
  mean  : 0.726
  std   : 0.026
  range : 0.688 to 0.773


In [18]:
# Train on two districts, evaluate HIGH-risk recall on the held-out third.
# This probes whether the weather-to-risk relationship transfers across districts,
# the core concern when generalising beyond the labelled set.
print("Leave-one-district-out transfer (XGBoost, HIGH-risk class)")
print("=" * 58)
for held_out in LABELLED_DISTRICTS:
    train = labelled[labelled["district"] != held_out]
    test  = labelled[labelled["district"] == held_out]
    model = build_models(train[TARGET_COL].values)["XGBoost"]
    model.fit(train[FEATURE_COLS].values, train[TARGET_COL].values)
    preds = model.predict(test[FEATURE_COLS].values)
    recall = recall_score(test[TARGET_COL].values, preds, zero_division=0)
    prec   = precision_score(test[TARGET_COL].values, preds, zero_division=0)
    print(f"  hold out {held_out:10s}: recall {recall:.3f}, precision {prec:.3f}")


Leave-one-district-out transfer (XGBoost, HIGH-risk class)


  hold out Arusha    : recall 0.710, precision 0.727


  hold out Mbeya     : recall 0.752, precision 0.728


  hold out Morogoro  : recall 0.728, precision 0.758


## 9. Risk Prediction Across All Districts

The trained XGBoost model is applied to every district to produce a risk
probability for each daily record, which is then banded into the HIGH / MEDIUM /
LOW levels the API serves.

### Scope constraint: trained on Arusha only

The disease labels exist for one district, Arusha, across two 2022 seasons.
Applying the model to the other 19 districts is an explicit extrapolation and a
known limitation of this capstone:

- **Agro-climatic difference.** The districts span five countries and a wide range
  of altitude and rainfall regimes (unimodal as well as bimodal). A weather to
  disease relationship calibrated in Arusha need not hold elsewhere.
- **Single-season, curated labels.** Labels derive from one location, two seasons,
  and a deliberately disease-rich image collection. They capture relative seasonal
  risk, not validated local incidence.
- **Weak supervision.** Labels are inferred from an aggregate class distribution
  plus a weather conduciveness model, not from per-field observations.

Predictions for non-Arusha districts should be read as screening signals and must
be validated against local ground truth before any operational use. Collecting
district-specific labels to close this gap is the priority for Sprint 2.


In [19]:
import joblib

# Probability bands mapped to the three risk levels the API returns.
RISK_BANDS = [(0.66, "HIGH"), (0.33, "MEDIUM"), (0.0, "LOW")]


def band_probability(prob: float) -> str:
    """Map a HIGH-risk probability to the API risk levels."""
    for threshold, level in RISK_BANDS:
        if prob >= threshold:
            return level
    return "LOW"


weather_features["risk_probability"] = xgb_model.predict_proba(
    weather_features[FEATURE_COLS].values
)[:, 1].round(3)
weather_features["risk_level"] = weather_features["risk_probability"].apply(band_probability)

# Persist the model and its feature order so the Sprint 2 API can load it directly.
MODEL_PATH = os.path.join(DRIVE_ROOT, "xgb_risk_model.joblib")
joblib.dump({"model": xgb_model, "features": FEATURE_COLS}, MODEL_PATH)

level_share = (
    weather_features.groupby("country")["risk_level"]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=["LOW", "MEDIUM", "HIGH"])
    .fillna(0.0)
    .round(3)
)
print(f"Model saved to: {MODEL_PATH}\n")
print("Risk-level share by country:")
print(level_share)


Model saved to: ./xgb_risk_model.joblib

Risk-level share by country:
risk_level    LOW  MEDIUM   HIGH
country                         
Ethiopia    0.196   0.280  0.524
Kenya       0.189   0.342  0.468
Rwanda      0.249   0.342  0.409
Tanzania    0.229   0.313  0.457
Uganda      0.316   0.328  0.355


In [20]:
fig, ax = plt.subplots(figsize=(11, 5))
level_share[["LOW", "MEDIUM", "HIGH"]].plot(
    kind="bar", stacked=True, ax=ax,
    color={"LOW": "#70AD47", "MEDIUM": "#E0A800", "HIGH": "#C0504D"},
)
ax.set_title("Predicted Risk-Level Distribution by Country", fontweight="bold")
ax.set_ylabel("Share of daily records")
ax.set_xlabel("")
ax.legend(title="Risk level", frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, "risk_by_country.png"), dpi=150, bbox_inches="tight")
plt.show()


## 10. Exploratory Analysis

Features are plotted at country level, one line per country averaged across its
four districts, to keep the charts readable.


In [21]:
COUNTRY_COLORS = {
    "Rwanda":   "#2E75B6",
    "Kenya":    "#C0504D",
    "Uganda":   "#70AD47",
    "Tanzania": "#E87722",
    "Ethiopia": "#7B4F9E",
}

country_avg = (
    weather_features
    .groupby(["country", "date"])[["consecutive_wet_days",
                                    "temp_spread_7d",
                                    "humidity_deviation"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)
fig.suptitle("AgroGuard: Engineered Features by Country (2021-2023)",
             fontsize=13, fontweight="bold", y=1.01)

PLOTS = [
    ("consecutive_wet_days", "Consecutive Wet Days"),
    ("temp_spread_7d",        "7-Day Temperature Spread (°C)"),
    ("humidity_deviation",    "Humidity Deviation from 7-Day Mean (%)"),
]

for ax, (feature, ylabel) in zip(axes, PLOTS):
    for country, color in COUNTRY_COLORS.items():
        subset = country_avg[country_avg["country"] == country]
        ax.plot(subset["date"], subset[feature],
                label=country, color=color, alpha=0.85, linewidth=0.9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(frameon=False, fontsize=9, loc="upper right")

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, "feature_distributions.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved.")

Plot saved.


In [22]:
# Summary statistics per country
(
    weather_features
    .groupby("country")[["consecutive_wet_days", "temp_spread_7d", "humidity_deviation"]]
    .agg(["mean", "std", "max"])
    .round(2)
)

consecutive_wet_days            temp_spread_7d               \
                         mean    std max           mean   std    max   
country                                                                
Ethiopia                 3.64  10.04  97          10.40  2.18  16.13   
Kenya                    1.28   2.63  24          11.17  1.85  17.47   
Rwanda                   2.02   3.84  40           9.75  1.17  13.51   
Tanzania                 2.04   5.05  40          10.25  2.15  15.86   
Uganda                   2.25   4.08  31           9.33  2.39  14.76   

         humidity_deviation               
                       mean   std    max  
country                                   
Ethiopia              -0.00  6.28  36.10  
Kenya                 -0.00  6.08  36.51  
Rwanda                -0.01  6.06  24.00  
Tanzania              -0.02  5.36  27.86  
Uganda                -0.02  6.00  37.67

In [23]:
# Correlation heatmaps per country
feature_cols     = ["rainfall", "consecutive_wet_days", "temp_spread_7d", "humidity_deviation"]
unique_countries = list(dict.fromkeys(d["country"] for d in DISTRICTS))

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
fig.suptitle("Feature Correlations by Country", fontsize=12, fontweight="bold")

for ax, country in zip(axes, unique_countries):
    subset = weather_features[weather_features["country"] == country][feature_cols]
    sns.heatmap(subset.corr(), ax=ax, annot=True, fmt=".2f",
                cmap="coolwarm", center=0, square=True,
                linewidths=0.5, cbar=False, annot_kws={"size": 8},
                vmin=-1, vmax=1)
    ax.set_title(country, fontsize=10, color=COUNTRY_COLORS.get(country, "#111827"))

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, "feature_correlations.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 11. Data Export


In [24]:
weather_features.to_csv(FEATURES_PATH, index=False)

print(f"Saved to        : {FEATURES_PATH}")
print(f"Total rows      : {len(weather_features):,}")
print(f"Districts       : {weather_features['district'].nunique()}")
print(f"Countries       : {weather_features['country'].nunique()}")
print(f"Date range      : {weather_features['date'].min().date()} to {weather_features['date'].max().date()}")
print(f"Features        : {', '.join(FEATURE_COLS)}")
print(f"Prediction cols : risk_probability, risk_level")


Saved to        : ./sprint1_features.csv
Total rows      : 21,900
Districts       : 20
Countries       : 5
Date range      : 2021-01-01 to 2023-12-31
Features        : consecutive_wet_days, temp_spread_7d, humidity_deviation, rainfall_7d_total
Prediction cols : risk_probability, risk_level
